In [10]:
class Value:
  
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"
  
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
      
    return out
  
  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out
  
  def __rmul__(self, other): # other * self
    return self * other

  def __truediv__(self, other): # self / other
    return self * other**-1

  def __neg__(self): # -self
    return self * -1

  def __sub__(self, other): # self - other
    return self + (-other)

  def __radd__(self, other): # other + self
    return self + other

  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')
    
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    
    return out
  
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')
    
    def _backward():
      self.grad += out.data * out.grad # NOTE: in the video I incorrectly used = instead of +=. Fixed here.
    out._backward = _backward
    
    return out
  
  
  def backward(self):
    
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [26]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [27]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

ModuleNotFoundError: No module named 'graphviz'

In [5]:
import torch

In [47]:
#usually the tensor stores the value only so we need requires_grad to calculate the gradient for this input
x1 = torch.Tensor([2.0]).double()                ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()                ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()               ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()                ; w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()  ; b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n) #neuron activation

print(o.data.item())
o.backward() # it does back propagation to calculate the gradient

print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

0.7071066904050358
---
x2 0.5000001283844369
w2 0.0
x1 -1.5000003851533106
w1 1.0000002567688737


In [84]:
class Neuron:
  
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))
  
  def __call__(self, x):
    # w * x + b
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out
  
  def parameters(self):
    return self.w + [self.b]

class Layer:
  
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs
  
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
  
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
  
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x
  
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

In [85]:
#forward propagation
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1]) #MLP(no.of.input, 1st layer neuron, 2nd layer neuron, last neuron)
n(x)

Value(data=0.4996985909247964)

In [86]:
len(n.parameters()) #returns the number of parameters including bias(b) as well

41

In [87]:
#we have four possible input sets to the neuron
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [88]:
# how to tune the weights to reach the desired target.
# we use to calculate a single number to understand the performance of the whole network i.e loss
# minimize the loss to predict the numbers 
#calculate the prediction
ypred = [n(x) for x in xs]
loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])
loss # lower the value higher the prediction rates

Value(data=6.3063448566221245)

In [91]:
for k in range(50):
  
  # forward pass
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  
  # backward pass
  for p in n.parameters():
    p.grad = 0.0 # the previous grad should be assigned zero if not the previous grad will keep added to the incoming gradient
  loss.backward()
  
  # update, gradient decent
  for p in n.parameters():
    p.data += -0.1 * p.grad # (-) used here to minimize the loss
    # the gradient points towards the increasing the loss
  
  print(k, loss.data)
  

0 0.036178806882726094
1 0.03295318656710964
2 0.03023411157755695
3 0.027912373153173062
4 0.02590794897372653
5 0.024160857644679817
6 0.022625258033970384
7 0.021265520476627917
8 0.020053539669579068
9 0.018966853541390532
10 0.017987299194942855
11 0.017100035068087972
12 0.01629281795077908
13 0.01555546061748107
14 0.014879419572984314
15 0.014257477929692267
16 0.01368349878388322
17 0.013152231484863268
18 0.012659158039850156
19 0.012200370294055246
20 0.011772470937657868
21 0.011372493126340049
22 0.010997834764652586
23 0.010646204430400742
24 0.010315576608634611
25 0.010004154421847067
26 0.009710338435129537
27 0.00943270041438588
28 0.009169961146015586
29 0.008920971604989223
30 0.008684696897569842
31 0.008460202514404089
32 0.008246642516257395
33 0.008043249343501407
34 0.007849324995519508
35 0.0076642333704667505
36 0.007487393591603978
37 0.007318274175488228
38 0.007156387921016411
39 0.007001287417755348
40 0.006852561087991071
41 0.006709829690152118
42 0.006

In [92]:
ypred

[Value(data=0.9622651181697386),
 Value(data=-0.9769404651478399),
 Value(data=-0.950988252155896),
 Value(data=0.9627309834327349)]